# fig_9_model_selection**Where in binding-rate space can the counts tell the two topologies apart?**`stochtf.scripts.run_inference --grid` fits the joint model -- monomer and
heterodimer at once, with the topology as a sampled parameter -- to synthetic
data at each point of a grid of binding rates. Every one of those datasets was
generated by the *heterodimer*, so a perfect discriminator would return
P(heterodimer) = 1 everywhere. It does not, and this figure is a map of where it
does and does not.

The Bayes factor is the honest quantity to plot rather than the raw posterior
probability, because it divides out the prior odds and so reports what the data
contributed. It is shown on a log scale symmetric about 1, where 1 means the
counts were silent on the question; on the Jeffreys reading anything below about
3 is not worth calling evidence.

Both are shown. The left panel is the Bayes factor, the right the posterior
probability on a plain 0-1 scale, where 0.5 is the prior and so marks the cells
the data could not decide.

**Usage**

    python -m stochtf.scripts.run_inference --grid --stride 16    # produce the matrix
    python figures/fig_9_model_selection.py               # plot it

In [ ]:
%matplotlib inline

import argparse
import os
import numpy as np
import matplotlib as mpl

In [ ]:
mpl.use("Agg")

import matplotlib.pyplot as plt

from matplotlib.colors import LogNorm

from stochtf import paths

from stochtf.plotting import PALETTE, output_path, use_paper_style

In [ ]:
#: Rates shown in the posterior panels, with the colour each is drawn in.
SHOWN = [("alpha_s", r"$k_{on,s}$", 0), ("alpha_n", r"$k_{on,n}$", 1),
         ("k_y", r"$k_y$", 2)]

In [ ]:
def load_posteriors(path):
    """Sampled posteriors for a few grid cells, or None if not generated."""
    if not os.path.exists(path):
        return None
    store = np.load(path, allow_pickle=False)
    return {k: store[k] for k in store.files}

In [ ]:
def load(path):
    if not os.path.exists(path):
        raise SystemExit(
            f"{path} not found. Produce it first with\n"
            "    python -m stochtf.scripts.run_inference --grid --stride 16")
    store = np.load(path, allow_pickle=False)
    return (store["p_heterodimer"], store["bayes_factor"], store["alpha"],
            int(store["stride"]), int(store["n_cells"]))

## Parameters

In [ ]:
# Parameters. These were command-line flags; edit them here.

matrix     = None  # grid .npz (default results/joint_grid_pheterodimer.npz)
posteriors = None  # sampled posteriors from stochtf.scripts.grid_sample_posteriors; the lower row is omitted when absent

In [ ]:
path = matrix or paths.results("joint_grid_pheterodimer.npz")

p_het, bayes, alpha, stride, n_cells = load(path)

posteriors = load_posteriors(posteriors or
                             paths.results("grid_sample_posteriors.npz"))

use_paper_style(sans_serif="Arial")

n_show = 0 if posteriors is None else len(posteriors["labels"])

if n_show:
    fig = plt.figure(figsize=(11.2, 8.2))
    grid = fig.add_gridspec(2, max(n_show, 2), height_ratios=[1.35, 1.0],
                            hspace=0.34, wspace=0.42)
    axes = [fig.add_subplot(grid[0, :max(n_show, 2) // 2]),
            fig.add_subplot(grid[0, max(n_show, 2) // 2:])]
    axes[1].sharex(axes[0]); axes[1].sharey(axes[0])
    lower = [fig.add_subplot(grid[1, k]) for k in range(n_show)]
else:
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6), sharex=True,
                             sharey=True)
    axes, lower = list(axes), []

In [ ]:
# Cell edges, so pcolormesh puts each fitted point at a cell centre rather
# than at a corner -- the grid is coarse enough for that to matter.
step = np.log10(alpha[1] / alpha[0]) if alpha.size > 1 else 1.0

edges = 10 ** np.concatenate([[np.log10(alpha[0]) - step / 2],
                              np.log10(alpha) + step / 2])

A_s, A_n = np.meshgrid(edges, edges)

### Bayes factor, log-scaled and symmetric about 1

In [ ]:
ax = axes[0]

finite = bayes[np.isfinite(bayes) & (bayes > 0)]

span = max(float(np.nanmax(finite)) if finite.size else 10.0, 10.0)

im = ax.pcolormesh(A_s, A_n, np.ma.masked_invalid(bayes), cmap="viridis",
                   norm=LogNorm(vmin=1.0 / span, vmax=span),
                   shading="flat", rasterized=True)

cb = fig.colorbar(im, ax=ax, pad=.02, fraction=.046)

cb.set_label("Bayes factor, heterodimer / monomer", fontsize=8)

cb.ax.tick_params(labelsize=7)

### Posterior probability on a plain 0-1 scale

In [ ]:
ax = axes[1]

im2 = ax.pcolormesh(A_s, A_n, np.ma.masked_invalid(p_het), cmap="viridis",
                    vmin=0.0, vmax=1.0, shading="flat", rasterized=True)

cb2 = fig.colorbar(im2, ax=ax, pad=.02, fraction=.046)

cb2.set_label("P(heterodimer | counts)", fontsize=8)

cb2.ax.tick_params(labelsize=7)

if n_show:
    for k in range(n_show):
        a_s = posteriors["true_alpha_s"][k]
        a_n = posteriors["true_alpha_n"][k]
        for ax in axes:
            ax.plot(a_s, a_n, "o", ms=8, mfc="w", mec="k", mew=1.2,
                    zorder=6)
            ax.annotate(str(k + 1), xy=(a_s, a_n), fontsize=6.5,
                        ha="center", va="center", zorder=7)

for ax in axes:
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$k_{on,s}$  (s$^{-1}$)")

axes[0].set_ylabel(r"$k_{on,n}$  (s$^{-1}$)")

In [ ]:
# Posterior marginals at the marked cells. Every rate is in units of
# gamma, so one shared log axis carries all three.
for k, ax in enumerate(lower):
    for name, label, colour in SHOWN:
        sample = posteriors[f"draws_{name}"][k]
        sample = sample[sample > 0]
        if sample.size == 0:
            continue
        bins = np.logspace(np.log10(sample.min()) - 0.15,
                           np.log10(sample.max()) + 0.15, 40)
        density, edges = np.histogram(sample, bins=bins, density=True)
        centres = np.sqrt(edges[:-1] * edges[1:])
        ax.plot(centres, density / density.max(), color=PALETTE[colour],
                lw=1.3, label=label)
        ax.fill_between(centres, density / density.max(), color=
                        PALETTE[colour], alpha=.18)

    truth = {"alpha_s": posteriors["true_alpha_s"][k],
             "alpha_n": posteriors["true_alpha_n"][k],
             "k_y": float(posteriors["k_y_true"])}
    for name, _, colour in SHOWN:
        ax.axvline(truth[name], color=PALETTE[colour], lw=1.0, ls="--",
                   alpha=.9)

    ax.set_xscale("log")
    ax.set_xlim(1e-4, 1e3)
    ax.set_ylim(0, 1.25)
    ax.set_yticks([])
    ax.set_xlabel(r"rate  ($\gamma$)")
    ax.text(0.03, 0.97,
            f"{k + 1}. {posteriors['labels'][k]}\n"
            f"BF {posteriors['bayes_factor'][k]:.2f}, "
            f"P(het) {posteriors['p_heterodimer'][k]:.2f}",
            transform=ax.transAxes, fontsize=6, va="top")
    if k == 0:
        ax.legend(fontsize=6, loc="upper right")
        ax.set_ylabel("posterior (scaled)")

total = int(np.isfinite(p_het).sum())

fig.savefig(output_path("fig_9_model_selection.svg"), bbox_inches="tight",
            facecolor="w")

In [ ]:
fig.savefig(output_path("fig_9_model_selection.png"), bbox_inches="tight",
            facecolor="w")

In [ ]:
print(f"Bayes factor: min {np.nanmin(bayes):.2f}  "
      f"median {np.nanmedian(bayes):.2f}  max {np.nanmax(bayes):.2f}")

print(f"P(heterodimer): min {np.nanmin(p_het):.3f}  "
      f"median {np.nanmedian(p_het):.3f}  max {np.nanmax(p_het):.3f}")

print(f"substantial evidence (BF > 3): "
      f"{int(np.nansum(bayes > 3))}/{total} points")